# Load and tokenize reviews

Load all reviews from the small cache (`data/reviews`). Each game maps to a list
of reviews, and each review is a list of lowercase word tokens.


In [1]:
MIN_REVIEWS_PER_GAME = 5
MIN_GAMES = 5
MAX_ASPECT_SHARE = 0.10
TOP_BIGRAMS = 3

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src.csv_io import REVIEWS_DATA_DIR, load_reviews
from src.review_parser import ReviewParser

parser = ReviewParser()


In [2]:
game_reviews = {
    int(path.stem): [parser.tokenize([review.review])[0]
                     for review in load_reviews(int(path.stem))]
    for path in sorted((p for p in REVIEWS_DATA_DIR.glob("*.csv") if p.stem.isdigit()),
                       key=lambda p: int(p.stem))
}



## Words supported across games

Include a word when it appears in at least `MIN_REVIEWS_PER_GAME` reviews
in each of at least `MIN_GAMES` games. Count each word once per review.


In [3]:
from collections import Counter

game_word_counts = {}
word_game_counts = Counter()
for game_id, reviews in game_reviews.items():
    review_counts = Counter(word for review in reviews for word in set(review))
    game_word_counts[game_id] = {word: count for word, count in review_counts.items()
                                 if count >= MIN_REVIEWS_PER_GAME}
    word_game_counts.update(game_word_counts[game_id].keys())

words = {word for word, count in word_game_counts.items() if count >= MIN_GAMES}
len(words)


3993

## Per-game word counts and IDF

`game_word_counts[game_id][word]` counts distinct reviews containing a vocabulary
word, retaining only counts ≥ `MIN_REVIEWS_PER_GAME`.
IDF uses the natural logarithm: `log(total games / qualifying games)`.
A qualifying game mentions the word in at least `MIN_REVIEWS_PER_GAME` reviews.


In [4]:
import math

# Reuse the review counts above; retain only vocabulary words.
game_word_counts = {
    game_id: {word: count for word, count in counts.items() if word in words}
    for game_id, counts in game_word_counts.items()
}
idf = {word: math.log(len(game_reviews) / word_game_counts[word]) for word in words}
idf


{'details': 2.3989277974125596,
 'poe': 4.478369339092396,
 'per': 2.494237977216885,
 'name': 2.166734410578432,
 'cavalry': 4.5736795188967205,
 'p2w': 4.103675889650985,
 'auction': 5.084505142662711,
 'next': 0.5218945710934157,
 'armour': 5.266826699456666,
 'lose': 1.7346010553881064,
 'hard': 0.1620940819812946,
 'leader': 3.6181680738692843,
 'score': 2.457424004094168,
 'heretic': 5.266826699456666,
 'research': 2.9844443137801395,
 'chasing': 5.084505142662711,
 'factor': 5.084505142662711,
 'matter': 1.209837923778334,
 'selection': 3.7852221585324504,
 'minor': 0.8898126066063284,
 'six': 4.5736795188967205,
 '30fps': 5.084505142662711,
 'solar': 4.79682307021093,
 'forza': 4.79682307021093,
 'information': 2.585805170742375,
 'handful': 4.930354462835453,
 'youtube': 2.7491302268456748,
 'gordon': 4.391357962102766,
 'peaceful': 4.679040034554547,
 'starvation': 5.266826699456666,
 'group': 2.0722435671575097,
 'dogs': 4.311315254429229,
 'attacked': 4.391357962102766,
 'a

## Games with low aspect coverage

Keep games where no aspect from `src/aspects.py` appears in more than
`MAX_ASPECT_SHARE` of reviews. Count each aspect once per review using the
original tokens, independently of the filtered vocabulary. Skip games with no reviews.


In [5]:
from src.aspects import ASPECTS

aspect_words = {aspect: set(vocabulary) for aspect, vocabulary in ASPECTS.items()}
aspect_shares = {}
for game_id, reviews in game_reviews.items():
    if not reviews:
        continue
    mentions = Counter()
    for review in reviews:
        review_words = set(review) | {a + " " + b for a, b in zip(review, review[1:])}
        mentions.update(aspect for aspect, vocabulary in aspect_words.items()
                        if review_words & vocabulary)
    aspect_shares[game_id] = {aspect: mentions[aspect] / len(reviews)
                              for aspect in aspect_words}

low_coverage_games = [game_id for game_id, shares in aspect_shares.items()
                      if max(shares.values(), default=0) <= MAX_ASPECT_SHARE]
len(low_coverage_games)


16

## Top TF-IDF words for five random low-coverage games

TF is the number of reviews mentioning the word divided by the game's total
review count. Multiply by the existing IDF and show the 10 highest scores.
Use all available low-coverage games if fewer than five qualify.


In [6]:
import random

selected_games = random.sample(low_coverage_games, min(5, len(low_coverage_games)))
for game_id in selected_games:
    scores = {word: count / len(game_reviews[game_id]) * idf[word]
              for word, count in game_word_counts[game_id].items()}
    top_words = sorted(scores, key=lambda word: (-scores[word], word))[:10]
    print(f"\nGame {game_id}")
    for word in top_words:
        print(f"{word:20} {scores[word]:.4f}")



Game 437900
for                  0.0115
it                   0.0078
game                 0.0031

Game 233720
medical              0.5320
hammer               0.5199
simulator            0.4801
controls             0.3832
heart                0.3459
hand                 0.2834
skull                0.2660
laugh                0.2450
silly                0.2416
comedy               0.2310

Game 1343370
best                 0.0104
your                 0.0071
up                   0.0071
play                 0.0068
can                  0.0055
not                  0.0054
in                   0.0050
with                 0.0049
have                 0.0048
but                  0.0044

Game 389570
io                   0.6362
fun                  0.0118
best                 0.0070
all                  0.0052
you                  0.0051
its                  0.0048
of                   0.0046
there                0.0044
great                0.0042
time                 0.0042

Game 1343400
coming   

## Top words shared by low-coverage games

Map each word to games having it among their top 20 TF-IDF words after
excluding existing aspect keywords and the parser's basic stop words.
Use all low-coverage games, and display words shared by at least two games,
ordered by game count.


In [7]:
excluded_words = set().union(*aspect_words.values()) | parser.stop_words

word_games = {}
for game_id in low_coverage_games:
    scores = {word: count / len(game_reviews[game_id]) * idf[word]
              for word, count in game_word_counts[game_id].items()
              if word not in excluded_words}
    top_words = sorted(scores, key=lambda word: (-scores[word], word))[:5]
    for word in top_words:
        word_games.setdefault(word, []).append(game_id)

word_games = dict(sorted(word_games.items(), key=lambda item: (-len(item[1]), item[0])))
for word, games in word_games.items():
    if len(games) >= 2:
        print(f"{word:20} {len(games):3} games: {', '.join(map(str, games))}")


game                   4 games: 207230, 437900, 1049590, 1536610
addicting              2 games: 346900, 424280
best                   2 games: 389570, 1343370
but                    2 games: 207230, 1343370
clicking               2 games: 346900, 363970
free                   2 games: 312990, 424280
fun                    2 games: 312990, 389570
idle                   2 games: 346900, 363970
play                   2 games: 312990, 1343370
players                2 games: 17080, 335240


## Separate bigram counts and IDF

Bigrams are adjacent token pairs within each review, counted once per review.
Use the same review/game support thresholds, with a separate vocabulary and IDF.
The existing token lists discard punctuation, so pairs can cross sentence boundaries.


In [8]:
game_bigram_counts = {}
bigram_game_counts = Counter()
for game_id, reviews in game_reviews.items():
    review_counts = Counter(pair for review in reviews
                            for pair in set(zip(review, review[1:])))
    game_bigram_counts[game_id] = {
        pair: count for pair, count in review_counts.items()
        if count >= MIN_REVIEWS_PER_GAME
    }
    bigram_game_counts.update(game_bigram_counts[game_id].keys())

bigrams = {pair for pair, count in bigram_game_counts.items() if count >= MIN_GAMES}
game_bigram_counts = {
    game_id: {pair: count for pair, count in counts.items() if pair in bigrams}
    for game_id, counts in game_bigram_counts.items()
}
bigram_idf = {pair: math.log(len(game_reviews) / bigram_game_counts[pair])
              for pair in bigrams}
len(bigrams)


3963

## Separate bigram TF-IDF and reverse map

TF = reviews containing the pair / total game reviews. Multiply by bigram IDF.
For ranking, exclude pairs containing basic stop words or exactly matching an
existing aspect keyword. Keep `TOP_BIGRAMS` (3) per game with reviews in
`bigram_games`, separately from `word_games`. Display pairs shared by at least
two games, ordered by game count.


In [9]:
bigram_scores = {
    game_id: {pair: count / len(game_reviews[game_id]) * bigram_idf[pair]
              for pair, count in game_bigram_counts[game_id].items()}
    for game_id, reviews in game_reviews.items() if reviews
}
bigram_games = {}
for game_id, scores in bigram_scores.items():
    candidates = [pair for pair in scores
                  if not (set(pair) & parser.stop_words)
                  and " ".join(pair) not in excluded_words]
    top_bigrams = sorted(candidates, key=lambda pair: (-scores[pair], pair))[:TOP_BIGRAMS]
    for pair in top_bigrams:
        bigram_games.setdefault(pair, []).append(game_id)

bigram_games = dict(sorted(bigram_games.items(), key=lambda item: (-len(item[1]), item[0])))
for pair, games in bigram_games.items():
    if len(games) >= 2:
        print(f"{' '.join(pair):30} {len(games):3} games: {', '.join(map(str, games))}")


early access                    39 games: 228380, 232090, 236110, 261550, 280790, 310560, 346010, 361420, 387990, 427730, 509980, 611500, 671860, 681660, 739630, 755790, 809960, 892970, 955050, 961200, 1017180, 1044720, 1145350, 1203620, 1217060, 1307550, 1363080, 1366540, 1371980, 1372880, 1491000, 1604030, 1621690, 1623730, 1832640, 2012510, 2479810, 2694490, 3164500
first game                      34 games: 8850, 17330, 20920, 35720, 67370, 108800, 202750, 232090, 238370, 274170, 324800, 356190, 388410, 403640, 409720, 414700, 447040, 454650, 532210, 534380, 554620, 560130, 601430, 612880, 641990, 728880, 848450, 996580, 1057090, 1145350, 1282100, 1771300, 1832640, 2054970
open world                      31 games: 8190, 39500, 50130, 55110, 55230, 200260, 202170, 206420, 225540, 234140, 243470, 249130, 287700, 307690, 355840, 360430, 412020, 460930, 637650, 668580, 851850, 905370, 990080, 1151640, 1211020, 1245620, 1259420, 1284210, 1372880, 1475810, 2215430
good game               